<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 3 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">模型语义与分区分桶</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">使用统一订单样本，观察 SQL、结果与验收证据。请按顺序运行单元。</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">Doris 4.1.3 target · Order data · Isolated course database</span>
</div>

By the end of this lab, you will have a running Doris environment, an `events` table containing more than 10 million rows, and analytical results produced from that table. Run the cells in order.

[讲义](course3_models_partitioning_and_bucketing.md) · [课程入口](../README.md)


## 实验范围

仅重建 d03_dup、d03_unique、d03_agg、d03_partitioned。重复 Key 的语义和数据分布是两个独立问题。


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course import WarehouseLab
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows
from dw_course.ui import show_sql, show_response

lab = WarehouseLab()




## 1. 相同键，不同模型

逐次插入 (1,100)、(1,50)、(2,200)。Duplicate 保留三行，Unique 最终保留两行，Aggregate 按 Key 累加金额。


In [ ]:
definitions = {
    "d03_dup": 'CREATE TABLE d03_dup (id BIGINT, amount DECIMAL(12,2)) DUPLICATE KEY(id) DISTRIBUTED BY HASH(id) BUCKETS 1 PROPERTIES("replication_num"="1")',
    "d03_unique": 'CREATE TABLE d03_unique (id BIGINT, amount DECIMAL(12,2)) UNIQUE KEY(id) DISTRIBUTED BY HASH(id) BUCKETS 1 PROPERTIES("replication_num"="1", "enable_unique_key_merge_on_write"="true")',
    "d03_agg": 'CREATE TABLE d03_agg (id BIGINT, amount DECIMAL(12,2) SUM) AGGREGATE KEY(id) DISTRIBUTED BY HASH(id) BUCKETS 1 PROPERTIES("replication_num"="1")',
}
for table, ddl in definitions.items():
    lab.execute(f"DROP TABLE IF EXISTS {table}")
    show_sql("建表 SQL", ddl)
    lab.execute(ddl)
    for row in [(1, "100.00"), (1, "50.00"), (2, "200.00")]:
        lab.insert(table, ["id", "amount"], [row])
expect(lab.query("SELECT id, amount FROM d03_dup ORDER BY id, amount"), [(1,"50.00"),(1,"100.00"),(2,"200.00")])
expect(lab.query("SELECT id, amount FROM d03_unique ORDER BY id"), [(1,"50.00"),(2,"200.00")])
expect(lab.query("SELECT id, amount FROM d03_agg ORDER BY id"), [(1,"150.00"),(2,"200.00")])


## 2. 分区与分桶各管什么

日期分区可以缩小扫描的分区范围，Hash 分桶负责分布。此演示表不承担订单当前状态，不能把它的复合排序键照搬成业务唯一键。


In [ ]:
lab.execute("DROP TABLE IF EXISTS d03_partitioned")
lab.execute("""
CREATE TABLE d03_partitioned (
    order_date DATE, order_id BIGINT, amount DECIMAL(12,2)
) DUPLICATE KEY(order_date, order_id)
PARTITION BY RANGE(order_date) (
    PARTITION p_day1 VALUES [('2026-01-01'), ('2026-01-02')),
    PARTITION p_day2 VALUES [('2026-01-02'), ('2026-01-03'))
)
DISTRIBUTED BY HASH(order_id) BUCKETS 4
PROPERTIES("replication_num"="1")
""")
lab.insert("d03_partitioned", ["order_date", "order_id", "amount"],
           [("2026-01-01",1001,"100.00"),("2026-01-02",1011,"110.00")])
for query in [
    "SELECT * FROM d03_partitioned",
    "SELECT * FROM d03_partitioned WHERE order_date = '2026-01-01'",
    "SELECT * FROM d03_partitioned WHERE order_date = '2026-01-01' AND order_id = 1001",
]:
    print(query)
    lab.sql("EXPLAIN " + query);
expect(lab.query("SELECT order_id, amount FROM d03_partitioned WHERE order_date = '2026-01-01'"), [(1001,"100.00")])
lab.close()


## 完成标准

指出三种模型的结果差异，并在计划中找到所选分区/Tablet 的证据。计划文本随版本变化，不以整段文本逐字相同验收。大数据扫描与索引实验留待补充。
